# Training on Google Colab (GPU)

This notebook trains every model used in the report on a free Colab GPU.
Training on a laptop CPU takes ~45 min per epoch at 640 px; a Colab T4 GPU takes ~1 min.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

Each finished run is copied to `MyDrive/mma3001_runs/` immediately, so if Colab
disconnects you only lose the run that was in progress. Re-running the notebook
skips runs that are already saved.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/Pork Rasher Error-Packaging-.v4i.yolov8.zip'   # Roboflow YOLOv8 export
SAVE = '/content/drive/MyDrive/mma3001_runs'                                  # where results are kept

In [ ]:
%cd /content
!rm -rf mma3001-pork-rasher-packing
!git clone -q https://github.com/noahlayton/mma3001-pork-rasher-packing.git
%cd /content/mma3001-pork-rasher-packing
!pip install -q ultralytics
!pip install -q -e .
# -o: the Roboflow zip contains data.yaml three times; overwrite silently
!mkdir -p data/pork-rasher-v4 && unzip -o -q "$ZIP" -d data/pork-rasher-v4
!ls data/pork-rasher-v4 && cat data/pork-rasher-v4/data.yaml

## 2. Experiments

| Run | Purpose |
|---|---|
| `y8n_640` | **main model** – YOLOv8 nano at the default 640 px |
| `y8s_640` | alternative – 3.6x more parameters: does a bigger model help? |
| `y8n_320` | input-resolution sensitivity – half resolution (4x fewer FLOPs) |
| `y8n_960` | input-resolution sensitivity – higher resolution for the tiny `packaging-error` boxes (optional, slowest) |
| `y8n_640_f25`, `y8n_640_f50` | data-requirement study – 25% / 50% of training images (optional) |

All runs: 50 epochs max, early stopping after 15 epochs without improvement, seed 0.

In [ ]:
import os, shutil, subprocess

RUNS = [  # (name, weights, imgsz, batch, fraction)
    ('y8n_640', 'yolov8n.pt', 640, 16, 1.0),
    ('y8s_640', 'yolov8s.pt', 640, 16, 1.0),
    ('y8n_320', 'yolov8n.pt', 320, 32, 1.0),
    ('y8n_960', 'yolov8n.pt', 960,  8, 1.0),
    ('y8n_640_f25', 'yolov8n.pt', 640, 16, 0.25),
    ('y8n_640_f50', 'yolov8n.pt', 640, 16, 0.5),
]
os.makedirs(SAVE, exist_ok=True)

for name, weights, imgsz, batch, frac in RUNS:
    if os.path.exists(f'{SAVE}/{name}/weights/best.pt'):
        print(f'skip {name}: already saved'); continue
    print(f'===== {name} =====')
    subprocess.run(['python', 'scripts/train.py', '--model', weights, '--imgsz', str(imgsz),
                    '--batch', str(batch), '--fraction', str(frac), '--epochs', '50',
                    '--device', '0', '--name', name], check=True)
    for d in (name, f'{name}_test'):
        shutil.copytree(f'runs/{d}', f'{SAVE}/{d}', dirs_exist_ok=True)
    print(f'saved {name} to Drive')

## 3. GPU speed benchmark

In [ ]:
import glob
w = sorted(glob.glob(f'{SAVE}/y8*/weights/best.pt'))
!python scripts/benchmark.py {' '.join(w)} --device 0 --out {SAVE}/benchmark_gpu.csv
# the main model at several inference sizes (same weights, different input resolution)
!python scripts/benchmark.py {SAVE}/y8n_640/weights/best.pt --imgsz 320 480 640 800 960 --device 0 --out {SAVE}/benchmark_gpu_imgsz.csv

## 4. Package the results

Creates one zip in your Drive (`MyDrive/mma3001_runs.zip`, ~150 MB). `last.pt` checkpoints are left out because only `best.pt` is used.
Share it as *Anyone with the link* so the results can be pulled into the analysis environment.

In [ ]:
import zipfile, pathlib
src = pathlib.Path(SAVE)
with zipfile.ZipFile(f'{SAVE}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in src.rglob('*'):
        if f.is_file() and f.name != 'last.pt':
            z.write(f, f.relative_to(src.parent))
print(f'{SAVE}.zip', round(pathlib.Path(f'{SAVE}.zip').stat().st_size / 2**20), 'MB')